In [1]:
from dotenv import load_dotenv
import os
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage, ToolMessage
from operator import add as add_messages
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.tools import tool

/tmp/ipykernel_9838/1397307779.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
load_dotenv()

True

In [3]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)  # I want to minimize hallucination - temperature = 0 makes the model output more deterministic 

In [4]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
pdf_path = "Stock_Market_Performance_2024.pdf"


# Safety measure I have put for debugging purposes :)
if not os.path.exists(pdf_path):
    raise FileNotFoundError(f"PDF file not found: {pdf_path}")

In [6]:
pdf_loader = PyPDFLoader(pdf_path) # This loads the PDF

In [7]:
# Checks if the PDF is there
try:
    pages = pdf_loader.load()
    print(f"PDF has been loaded and has {len(pages)} pages")
except Exception as e:
    print(f"Error loading PDF: {e}")
    raise

PDF has been loaded and has 9 pages


In [11]:
# Chunking Process
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

In [12]:
pages_split = text_splitter.split_documents(pages) # We now apply this to our pages

In [17]:
persist_directory = "./chroma_db"
collection_name = "stock_market"

# If our collection does not exist in the directory, create it
if not os.path.exists(persist_directory):
    os.makedirs(persist_directory)

try:
    # Create the Chroma database
    vectorstore = Chroma.from_documents(
        documents=pages_split,
        embedding=embeddings,
        persist_directory=persist_directory,
        collection_name=collection_name
    )

    print("Created ChromaDB vector store!")

except Exception as e:
    print(f"Error setting up ChromaDB: {e}")
    raise

Created ChromaDB vector store!


In [18]:
# Now we create our retriever 
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5} # K is the amount of chunks to return
)

In [19]:
@tool
def retriever_tool(query: str) -> str:
    """
    This tool searches and returns the information from the Stock Market Performance 2024 document.
    """

    docs = retriever.invoke(query)

    if not docs:
        return "I found no relevant information in the Stock Market Performance 2024 document."
    
    results = []
    for i, doc in enumerate(docs):
        results.append(f"Document {i+1}:\n{doc.page_content}")
    
    return "\n\n".join(results)


tools = [retriever_tool]

llm = llm.bind_tools(tools)

In [20]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]


def should_continue(state: AgentState):
    """Check if the last message contains tool calls."""
    result = state['messages'][-1]
    return hasattr(result, 'tool_calls') and len(result.tool_calls) > 0

In [21]:
system_prompt = """
You are an intelligent AI assistant who answers questions about Stock Market Performance in 2024 based on the PDF document loaded into your knowledge base.
Use the retriever tool available to answer questions about the stock market performance data. You can make multiple calls if needed.
If you need to look up some information before asking a follow up question, you are allowed to do that!
Please always cite the specific parts of the documents you use in your answers.
"""

In [22]:
tools_dict = {our_tool.name: our_tool for our_tool in tools} # Creating a dictionary of our tools

In [23]:
# LLM Agent
def call_llm(state: AgentState) -> AgentState:
    """Function to call the LLM with the current state."""
    messages = list(state['messages'])
    messages = [SystemMessage(content=system_prompt)] + messages
    message = llm.invoke(messages)
    return {'messages': [message]}


# Retriever Agent
def take_action(state: AgentState) -> AgentState:
    """Execute tool calls from the LLM's response."""

    tool_calls = state['messages'][-1].tool_calls
    results = []
    for t in tool_calls:
        print(f"Calling Tool: {t['name']} with query: {t['args'].get('query', 'No query provided')}")
        
        if not t['name'] in tools_dict: # Checks if a valid tool is present
            print(f"\nTool: {t['name']} does not exist.")
            result = "Incorrect Tool Name, Please Retry and Select tool from List of Available tools."
        
        else:
            result = tools_dict[t['name']].invoke(t['args'].get('query', ''))
            print(f"Result length: {len(str(result))}")
            

        # Appends the Tool Message
        results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))

    print("Tools Execution Complete. Back to the model!")
    return {'messages': results}


In [24]:
graph = StateGraph(AgentState)
graph.add_node("llm", call_llm)
graph.add_node("retriever_agent", take_action)

graph.add_conditional_edges(
    "llm",
    should_continue,
    {True: "retriever_agent", False: END}
)
graph.add_edge("retriever_agent", "llm")
graph.set_entry_point("llm")

rag_agent = graph.compile()

In [ ]:
def running_agent():
    print("\n=== RAG AGENT===")
    
    while True:
        user_input = input("\nWhat is your question: ")
        if user_input.lower() in ['exit', 'quit']:
            break
            
        messages = [HumanMessage(content=user_input)] # converts back to a HumanMessage type

        result = rag_agent.invoke({"messages": messages})
        
        print("\n=== ANSWER ===")
        print(result['messages'][-1].content)


running_agent()


=== RAG AGENT===



What is your question:  how was the SMP500 performing in 2024


Calling Tool: retriever_tool with query: SMP500 performance in 2024
Result length: 4385
Tools Execution Complete. Back to the model!

=== ANSWER ===
The S&P 500 performed remarkably well in 2024, delivering a roughly 25% total return for the year (around +23% in price terms) (Document 2). This marked the second consecutive year of over 20% returns for the S&P 500, a feat not observed since the late 1990s (Document 2). The tech-heavy Nasdaq Composite outpaced the broader market, jumping nearly 29% for the year (Document 2). However, smaller-cap stocks had more modest performance, with the S&P 500 Equal-Weight index and the Russell 2000 (small-cap benchmark) each rising about 10-11% in 2024 (Document 2). The dominance of mega-cap technology stocks was a key theme in 2024, with the "Magnificent 7" companies (Apple, Microsoft, Alphabet, Amazon, Meta, Nvidia, and Tesla) collectively surging by roughly 64-67% on average in 2024 (Document 4). These few firms contributed disproportionately to 


What is your question:  how did openai perform in 2024 


Calling Tool: retriever_tool with query: OpenAI stock market performance 2024
Result length: 4354
Tools Execution Complete. Back to the model!

=== ANSWER ===
Based on the provided documents, OpenAI is not explicitly mentioned as a performer in the stock market in 2024. However, the documents do mention the AI boom and its impact on various tech stocks, including Nvidia and Arm. It is possible that OpenAI may have been mentioned in the document, but it is not present in the provided snippets.
